In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
:set -XGADTs
:set -XDataKinds
:set -XTypeFamilies
:set -XPolyKinds
:set -XMultiParamTypeClasses
:set -XFunctionalDependencies
:set -XInstanceSigs
import Data.List (nub, sort, intercalate, (\\))
import Data.Maybe (fromMaybe)
import qualified Data.Map.Strict as Map
import Data.Map.Strict (Map)
import qualified Data.Set as Set
import Data.Set (Set)
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Toposes, Sheaves, Subobject Classifier, Internal Logic"

✅ SETUP OK: Toposes, Sheaves, Subobject Classifier, Internal Logic

# 🌏 Топосы, Пучки и Логика Классификатора Подобъектов

> **Девиз:** Топос — категория, которая ведёт себя как категория множеств, но живёт в произвольном математическом мире.

**Источники:** Johnstone «Sketches of an Elephant» (2002); Mac Lane & Moerdijk «Sheaves in Geometry and Logic» (1992); Lambek & Scott «Intro to Higher-Order Categorical Logic» (1986); nLab.

---

## Содержание

| № | Раздел | Суть |
|---|--------|------|
| T0 | Мотивация: три лица топоса | Геометрия / Логика / Типы |
| T1 | Предпучки и пучки | Presheaves, склейка, сайт Гротендика |
| T2 | Элементарный топос: аксиомы | Пределы, степенные объекты, $\Omega$ |
| T3 | Классификатор подобъектов $\Omega$ | $\mathrm{Hom}(A,\Omega) \cong \mathrm{Sub}(A)$ |
| T4 | Внутренняя логика топоса | Гейтингова алгебра, интуиционизм |
| T5 | Пучки и Битопосы | Два классификатора, паранепротиворечивость |
| T6 | Геометрические морфизмы | $f^* \dashv f_*$, связь с расширениями Кана |
| T7 | Внутренний язык (Mitchell-Benabou) | Типы = объекты, формулы = стрелки к $\Omega$ |
| T8 | Итоги | Сводная таблица, связи |
| T9 | Вишенка: $\mathbf{BiTopos}$ | Два классификатора = две логики |
| T10 | Haskell и топосная структура | Библиотеки |

> **📦 Зависимости**
> **Пакеты:** `base`, `containers`
> **Расширения:** `DataKinds` — типы поднимаются в сорта (kinds) ([→](Extensions.ipynb#datakinds)) · `DeriveFunctor` — deriving Functor ([→](Extensions.ipynb#deriving)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `FunctionalDependencies` — зависимости параметров класса: | m -> s ([→](Extensions.ipynb#multiparamtypeclasses)) · `GADTs` — конструкторы с уточнёнными типами результата ([→](Extensions.ipynb#gadts)) · `GeneralizedNewtypeDeriving` — newtype наследует инстансы обёрнутого типа ([→](Extensions.ipynb#deriving)) · `InstanceSigs` — сигнатуры методов прямо в instance ([→](Extensions.ipynb#instancesigs)) · `MultiParamTypeClasses` — классы с несколькими параметрами ([→](Extensions.ipynb#multiparamtypeclasses)) · `OverloadedStrings` — строковые литералы любого типа IsString ([→](Extensions.ipynb#overloadedstrings)) · `PolyKinds` — полиморфизм по сортам ([→](Extensions.ipynb#datakinds)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables)) · `TypeFamilies` — функции на уровне типов ([→](Extensions.ipynb#typefamilies))


---

## T0 — Мотивация: три лица топоса

![Три лица топоса](../diagrams/topos/three_faces.svg)

**Геометрическое лицо** — категория пучков $\mathbf{Sh}(X)$. Пучок на $X$ описывает правило склейки локальных данных в глобальные.

**Логическое лицо** — **классификатор подобъектов** $\Omega$: морфизмы $A \to \Omega$ биективно описывают подобъекты. В $\mathbf{Set}$: $\Omega=\{\bot,\top\}$ — булева логика. В $\mathbf{Sh}(X)$: $\Omega$ — пучок открытых, логика интуиционистская.

**Типо-теоретическое лицо** — **внутренний язык** Mitchell–Benabou: объекты = типы, стрелки = функции, предикаты = $A \to \Omega$, кванторы $\forall,\exists$ = расширения Кана вдоль проекции.

| Категория | $\Omega$ | Логика |
|----------|------|--------|
| $\mathbf{Set}$ | $\{\bot,\top\}$ | Классическая |
| $\mathbf{Sh}(X)$ | $\mathcal{O}(-)$ | Интуиционистская |
| $[\mathcal{C},\mathbf{Set}]$ | предпучок $\Omega$ | Интуиционистская |
| $\mathbf{BiTopos}([0,1])$ | $\mathrm{Pl}$ и $\mathrm{Bel}$ | Субъективная |


In [2]:
-- T0: Базовые типы (используются во всех остальных разделах)
class HeytingAlgebra h where
  htop     :: h
  hbot     :: h
  hmeet    :: h -> h -> h
  hjoin    :: h -> h -> h
  himplies :: h -> h -> h
  hneg :: h -> h
  hneg a = himplies a hbot

newtype Term ctx a = Term { runTerm :: ctx -> a }
  deriving Functor
type Predicate ctx = Term ctx Bool

instance HeytingAlgebra Bool where
  htop = True
  hbot = False
  hmeet = (&&)
  hjoin = (||)
  himplies a b = not a || b

putStrLn "\x2705 T0: HeytingAlgebra Bool (classical logic):"
putStrLn $ "  excluded middle T: " ++ show (hjoin True  (hneg True))
putStrLn $ "  excluded middle F: " ++ show (hjoin False (hneg False))
putStrLn $ "  neg neg = id (Boolean): " ++ show (all (\b -> hneg (hneg b) == b) [True, False])

✅ T0: HeytingAlgebra Bool (classical logic):

  excluded middle T: True

  excluded middle F: True

  neg neg = id (Boolean): True

---

## 1️⃣ T1 — Предпучки и Пучки

![Диаграмма склейки пучка](../diagrams/topos/sheaf_gluing.svg)

### Предпучок

**Определение.** Пусть $\mathcal{C}$ — малая категория. **Предпучок** — контравариантный функтор:
$$F : \mathcal{C}^{\mathrm{op}} \to \mathbf{Set}$$
Для частного случая: $\mathcal{C} = \mathbf{Open}(X)$. Объекты — открытые множества, морфизмы — включения. $F(U)$ — множество **секций** над $U$. Для включения $V \hookrightarrow U$: функция **ограничения** $s \mapsto s|_V$.

**Пример.** $X = \mathbb{R}$, $F(U) = \mathcal{C}(U, \mathbb{R})$ — непрерывные функции. Ограничение $f|_V$ — буквально ограничение функции на подмножество.

### Условие склейки (пучковое)

Предпучок $F$ — **пучок**, если для любого покрытия $U = \bigcup_i U_i$ выполнено:

- **Уникальность:** если $s|_{U_i} = t|_{U_i}$ для всех $i$, то $s = t$
- **Склейка:** если есть секции $s_i \in F(U_i)$ с $s_i|_{U_i \cap U_j} = s_j|_{U_i \cap U_j}$, то существует $s \in F(U)$ с $s|_{U_i} = s_i$

### Категория предпучков $\widehat{\mathcal{C}} = [\mathcal{C}^{\mathrm{op}}, \mathbf{Set}]$

Все предпучки образуют **категорию** $\widehat{\mathcal{C}}$ — это **всегда** элементарный топос. Морфизм предпучков $\alpha : F \to G$ — естественное преобразование: семейство $\alpha_U : F(U) \to G(U)$, коммутирующее с ограничениями.

### Сайт Гротендика

**Сайт** $(\mathcal{C}, J)$ = категория + топология Гротендика $J$. Категория пучков $\mathbf{Sh}(\mathcal{C},J) \hookrightarrow \widehat{\mathcal{C}}$ — всегда топос. Примеры топосов: $\mathbf{Set} = \mathbf{Sh}(1)$; $\mathbf{Sh}(X)$; $[\mathcal{C},\mathbf{Set}]$ (тривиальная $J$); $\mathbf{B}G$ ($G$-множества).


In [3]:
-- T1: Предпучки, пучки, условие склейки
-- Открытое множество = Set Int (дискретное пространство X = {0..4})
type OpenSet = Set.Set Int

-- Секция предпучка F: здесь F(U) = функция U -> Double
newtype Sections = Sections { getSections :: Map.Map Int Double }
  deriving (Eq, Show)

-- Ограничение секции c U на V (V \\subseteq U)
restrictTo :: OpenSet -> Sections -> Sections
restrictTo v (Sections m) = Sections (Map.filterWithKey (\k _ -> Set.member k v) m)

-- Проверка условия склейки для X={0,1,2}, U1={0,1}, U2={1,2}
checkGluing :: Bool
checkGluing = compatible && gluedOk
  where
    u1  = Set.fromList [0,1]
    u2  = Set.fromList [1,2]
    u12 = Set.intersection u1 u2
    s1  = Sections (Map.fromList [(0, 10.0), (1, 20.0)])
    s2  = Sections (Map.fromList [(1, 20.0), (2, 30.0)])
    compatible = restrictTo u12 s1 == restrictTo u12 s2
    glued = Sections (Map.fromList [(0, 10.0), (1, 20.0), (2, 30.0)])
    gluedOk = restrictTo u1 glued == s1 && restrictTo u2 glued == s2

-- Предпучок Єнеды: h_c(U) = Hom(c, U) = 1 если c \\subseteq U, иначе 0
yonedaHom :: OpenSet -> OpenSet -> Int
yonedaHom c u = if Set.isSubsetOf c u then 1 else 0

putStrLn $ "\x2705 T1: Условие склейки: " ++ show checkGluing
putStrLn $ "Yoneda h_{0}({0,1}) = " ++ show (yonedaHom (Set.singleton 0) (Set.fromList [0,1]))
putStrLn $ "Yoneda h_{0}({1,2}) = " ++ show (yonedaHom (Set.singleton 0) (Set.fromList [1,2]))
putStrLn $ "Yoneda h_{0}({0})   = " ++ show (yonedaHom (Set.singleton 0) (Set.singleton 0))

✅ T1: Условие склейки: True

Yoneda h_{0}({0,1}) = 1

Yoneda h_{0}({1,2}) = 0

Yoneda h_{0}({0})   = 1

---

## 2️⃣ T2 — Элементарный Топос: Аксиомы

![Аксиомы элементарного топоса](../diagrams/topos/topos_axioms.svg)

### Определение (Лоэр, Тирни, 1969–1972)

**Элементарный топос** — категория $\mathcal{E}$ с тремя аксиомами:

**Аксиома 1: Конечные пределы.** Существуют все конечные пределы: терминальный объект $1$, произведения $A \times B$, уравнители, декартовы квадраты (pullback).

**Аксиома 2: Степенные объекты.** Для каждого $B$ есть функтор $(-)^B : \mathcal{E} \to \mathcal{E}$, правый сопряжённый к $(-) \times B$:
$$\mathrm{Hom}(A \times B, C) \cong \mathrm{Hom}(A, C^B)$$

**Аксиома 3: Классификатор подобъектов.** Существует $\Omega \in \mathcal{E}$ с $\mathrm{true} : 1 \to \Omega$, такие, что для каждого моно $m : S \hookrightarrow A$ существует единственный $\chi_m : A \to \Omega$ со square-пуллбачком:
$$S \xrightarrow{!} 1 \xleftarrow{\mathrm{true}} \Omega \xleftarrow{\chi_m} A$$

### Примеры

| Топос | $\Omega$ | Особенность |
|------|---------|----------|
| $\mathbf{Set}$ | $\{\bot,\top\}$ | Простые предикаты |
| $\mathbf{Sh}(X)$ | $U \mapsto \mathcal{O}(U)$ | Интуиционистская логика |
| $[\mathcal{C},\mathbf{Set}]$ | $c \mapsto \mathrm{Sub}(\mathrm{Hom}(-,c))$ | Естествен. преобр.
| $\mathbf{B}G$ | $\{\bot,\top\}$ с действ. | Булев |

### Геометрический морфизм (предварительно)

Морфизм топосов $f : \mathcal{F} \to \mathcal{E}$ — пара $f^* \dashv f_*$ где $f^*$ сохраняет конечные пределы. Подробно — в T6.


In [4]:
-- T2: Аксиомы элементарного топоса в Haskell
-- Аксиома 1: Конечные пределы (в Hask: (), (,), filter)
terminal :: a -> ()
terminal _ = ()

product2 :: a -> b -> (a, b)
product2 a b = (a, b)

equalizer :: Eq c => (a -> c) -> (a -> c) -> [a] -> [a]
equalizer f g = filter (\x -> f x == g x)

-- Аксиома 2: Степенные объекты = curry/uncurry
-- Hom(A x B, C) ~= Hom(A, C^B) = curry
type PowerObj b c = b -> c

curryIso :: ((a,b) -> c) -> (a -> b -> c)
curryIso = curry

uncurryIso :: (a -> b -> c) -> ((a,b) -> c)
uncurryIso = uncurry

-- Аксиома 3: Классификатор подобъектов (в Set: Bool)
data SubObject a where
  SubOf :: (a -> Bool) -> SubObject a

characteristic :: SubObject a -> (a -> Bool)
characteristic (SubOf p) = p

pullbackSub :: SubObject a -> [a] -> [a]
pullbackSub (SubOf p) = filter p

evenSub :: SubObject Int
evenSub = SubOf even

let dom = [0..9]
let evens = pullbackSub evenSub dom
putStrLn $ "\x2705 T2: SubObject (even) in [0..9]: " ++ show evens
putStrLn $ "characteristic(4)=" ++ show (characteristic evenSub 4)
putStrLn $ "characteristic(3)=" ++ show (characteristic evenSub 3)
putStrLn $ "curryIso: " ++ show (curryIso (\(x,y) -> x+y) 3 4 == (7 :: Int))
putStrLn $ "Hom(AxB,C) ~= Hom(A, C^B): curry/uncurry are iso"

✅ T2: SubObject (even) in [0..9]: [0,2,4,6,8]

characteristic(4)=True

characteristic(3)=False

curryIso: True

Hom(AxB,C) ~= Hom(A, C^B): curry/uncurry are iso

---

## 3️⃣ T3 — Классификатор Подобъектов $\Omega$

![Pullback-диаграмма классификатора](../diagrams/topos/omega_pullback.svg)

**Классификатор подобъектов** — объект $\Omega$ с морфизмом $\mathrm{true} : 1 \to \Omega$, реализующий биекцию:
$$\mathrm{Hom}_{\mathcal{E}}(A, \Omega) \cong \mathrm{Sub}_{\mathcal{E}}(A)$$
естественную по $A$. Здесь $\mathrm{Sub}(A)$ — множество классов изоморфизма моно $S \hookrightarrow A$.

### Pullback-определение

Для монического $m : S \hookrightarrow A$ **характеристическая стрелка** $\chi_m : A \to \Omega$ — единственная, делающая декартов квадрат:
$$\begin{array}{ccc} S & \xrightarrow{!_S} & 1 \\ {\scriptstyle m}\downarrow\phantom{m} & \square & \phantom{\mathrm{true}}\downarrow{\scriptstyle\mathrm{true}} \\ A & \xrightarrow{\chi_m} & \Omega \end{array}$$

То есть $S = \chi_m^{-1}(\mathrm{true})$ — «прообраз» элемента истины. Это обобщение индикаторной функции на произвольный топос.

### $\Omega$ в $\mathbf{Set}$

$\Omega = \{\bot, \top\}$, $\mathrm{true}(\bullet) = \top$. Подобъект $S \subseteq A$: $\chi_S(a) = \top \Leftrightarrow a \in S$. Булева алгебра. $|\mathrm{Sub}(A)| = 2^{|A|}$.

### $\Omega$ в $\mathbf{Sh}(X)$

Классификатор — **пучок открытых множеств** $\Omega_X$:
$$\Omega_X(U) = \{V \subseteq U \mid V \text{ открыто}\} = \mathcal{O}(U)$$
Ограничение: $W \mapsto W \cap V$. Элемент $\mathrm{true}$ в $\Omega_X(U)$ — это само $U$. Характеристическая стрелка подпучка $\mathcal{S} \hookrightarrow \mathcal{F}$: $\chi_U(s) = \bigcup\{V \subseteq U \mid s|_V \in \mathcal{S}(V)\}$.

### Алгебра на $\Omega$

В любом топосе $\Omega$ несёт структуру внутренней Гейтинговой алгебры: $\top, \bot, \wedge, \vee, \Rightarrow, \neg$ с $\neg a = a \Rightarrow \bot$. Важно: $\neg\neg \neq \mathrm{id}$ в общем топосе.


In [5]:
-- T3: Классификатор подобъектов
-- Omega(в Sh(X)): открытые подмножества U

-- Omega(U) = {V | V \\subseteq U} = множество всех подмножеств U
omegaAt :: OpenSet -> [OpenSet]
omegaAt u = map Set.fromList (powerSet (Set.toList u))
  where
    powerSet [] = [[]]
    powerSet (x:xs) = let rest = powerSet xs in rest ++ map (x:) rest

-- true: U \\to Omega(U) = U (максимальное открытое)
trueAt :: OpenSet -> OpenSet
trueAt u = u

-- Характеристическая функция подобъекта: chi_S(a) = T если a в S
-- В Set: Omega = Bool, characteristic = предикат
chiSet :: (a -> Bool) -> a -> Bool
chiSet predicate a = predicate a

-- В Sh(X): chi_S(s)(U) = большее V \\subseteq U такое что s|V в S(V)
-- (здесь моделируем дискретно: S задан предикатом на секциях)
chiSheaf :: (Double -> Bool) -> Map.Map Int Double -> OpenSet -> OpenSet
chiSheaf inSub sections u =
  Set.fromList [k | k <- Set.toList u,
                    case Map.lookup k sections of
                      Just v -> inSub v
                      Nothing -> False]

-- Пример: Sub = секции > 15, X = {0,1,2}
let secs = Map.fromList [(0, 10.0), (1, 20.0), (2, 30.0)] :: Map.Map Int Double
let inSubBig = (> 15.0)
let u = Set.fromList [0,1,2]
let chi = chiSheaf inSubBig secs u
putStrLn $ "\x2705 T3: chi_S(sections)(U) = " ++ show (Set.toList chi) ++ " <- точки, где s > 15"
putStrLn $ "true_U = " ++ show (Set.toList (trueAt u))
putStrLn $ "|Omega({0,1,2})| = " ++ show (length (omegaAt u)) ++ " (все подмножества)"
putStrLn $ "Set omega: Hom(A, Bool) ~= Sub(A): " ++ show (chiSet even 4) ++ " / " ++ show (chiSet even 3)

✅ T3: chi_S(sections)(U) = [1,2] <- точки, где s > 15

true_U = [0,1,2]

|Omega({0,1,2})| = 8 (все подмножества)

Set omega: Hom(A, Bool) ~= Sub(A): True / False

---

## 4️⃣ T4 — Внутренняя Логика Топоса

![Решётка значений истинности](../diagrams/topos/logic_lattice.svg)

### Гейтингова алгебра на $\Omega$

В любом топосе $\Omega$ автоматически несёт структуру **внутренней Гейтинговой алгебры**. Операции $\wedge, \vee, \Rightarrow, \neg$ определены как морфизмы в $\mathcal{E}$.

**Гейтингова алгебра** — решётка $(H, \leq)$ с операцией импликации $a \Rightarrow b$, удовлетворяющей:
$$x \leq a \Rightarrow b \iff x \wedge a \leq b$$
(т.е. $(- \wedge a) \dashv (a \Rightarrow -)$). Отрицание: $\neg a = a \Rightarrow \bot$.

### Связки как классифицирующие стрелки (стрелочная нотация)

В элементарном топосе **все** логические связки — это морфизмы на $\Omega$, заданные через
классификатор подобъектов (характеристические стрелки определяющих подобъектов).

**Истина и ложь.** $\mathrm{true} : 1 \to \Omega$ — структура классификатора; ложь — это
характеристическая стрелка моно из инициального объекта:
$$\mathrm{false} = \chi_{(0 \hookrightarrow 1)} : 1 \to \Omega.$$

**Конъюнкция** $\wedge : \Omega\times\Omega \to \Omega$ — характеристическая стрелка точки
«обе истинны» $\langle\mathrm{true},\mathrm{true}\rangle : 1 \hookrightarrow \Omega\times\Omega$:
$$\begin{array}{ccc}
1 & \xrightarrow{\;!\;} & 1 \\
{\scriptstyle\langle\mathrm{true},\mathrm{true}\rangle}\downarrow & \square & \downarrow{\scriptstyle\mathrm{true}} \\
\Omega\times\Omega & \xrightarrow{\;\wedge\;} & \Omega
\end{array}$$

**Внутренний порядок и импликация.** Подобъект ${\leq}\hookrightarrow \Omega\times\Omega$ —
это эквалайзер пары $(\wedge,\ \pi_1)$, то есть $\{(x,y)\mid x\wedge y = x\}$. Импликация —
его характеристическая стрелка:
$$\Rightarrow\ =\ \chi_{(\leq)} : \Omega\times\Omega \to \Omega.$$

**Дизъюнкция** $\vee : \Omega\times\Omega \to \Omega$ — характеристическая стрелка **образа**
объединения двух «прямых истинности»
$[\langle \mathrm{true}\circ\,!,\ \mathrm{id}\rangle,\ \langle \mathrm{id},\ \mathrm{true}\circ\,!\rangle] : \Omega \sqcup \Omega \to \Omega\times\Omega$:
$$\vee\ =\ \chi_{\mathrm{im}[\dots]}.$$

**Отрицание** $\neg : \Omega \to \Omega$ — характеристическая стрелка лжи, равносильно
$\neg = (\Rightarrow)\circ\langle\mathrm{id},\ \mathrm{false}\circ\,!\rangle$:
$$\neg\ =\ \chi_{(\mathrm{false} : 1 \hookrightarrow \Omega)}.$$

**Гейтингова сопряжённость.** Эти стрелки делают $\Omega$ внутренней гейтинговой алгеброй:
$(-\wedge a)\dashv(a\Rightarrow -)$, то есть $x \leq (a\Rightarrow y) \iff (x\wedge a)\leq y$.

![Конъюнкция и отрицание как классифицирующие стрелки](../diagrams/topos/topos_conj_neg.svg)

![Импликация (эквалайзер) и дизъюнкция (образ)](../diagrams/topos/topos_impl_disj.svg)

> **Связь с реализацией ниже.** Открытые формулы в таблице ($\cap,\ \cup,\ \mathrm{Int}((X\setminus U)\cup V),\ \mathrm{Int}(X\setminus U)$) — это те же стрелки $\wedge,\vee,\Rightarrow,\neg$, реализованные в топосе пучков $\mathbf{Sh}(X)$. В $\mathbf{Set}$ ($\Omega=2$) из тех же pullback-квадратов выпадают классические таблицы истинности — это проверяет код далее.

### Почему логика интуиционистская?

**Пример в $\mathbf{Sh}(\mathbb{R})$:** $U = (-1,0) \cup (0,1)$.
$$\neg U = \mathrm{Int}(\mathbb{R} \setminus U) = (-\infty,-1) \cup (1,+\infty)$$
$$\neg\neg U = \mathrm{Int}(\mathbb{R} \setminus \neg U) = \mathrm{Int}([-1,1]) = (-1,1)$$
Точка $0 \in \neg\neg U$, но $0 \notin U$. Следовательно, $\neg\neg U \neq U$.

**Оператор $\neg\neg$** — это оператор регуляризации: $\neg\neg V = \mathrm{Int}(\mathrm{Cl}(V))$. Это **модальный оператор** (топология Лоэра). Регулярные открытые образуют булеву алгебру — «путь к классике» внутри интуиционистского мира.

### Операции в $\Omega_{\mathbf{Sh}(X)}$

| Операция | Формула | Смысл |
|----------|---------|-------|
| $U \wedge V$ | $U \cap V$ | Конъюнкция |
| $U \vee V$ | $U \cup V$ | Дизъюнкция |
| $U \Rightarrow V$ | $\mathrm{Int}((X\setminus U) \cup V)$ | Импликация |
| $\neg U$ | $\mathrm{Int}(X \setminus U)$ | Отрицание |
| $\neg\neg U$ | $\mathrm{Int}(\mathrm{Cl}(U))$ | Регуляризация |
| $U \vee \neg U$ | $U \cup \mathrm{Int}(X \setminus U)$ | Не всегда $X$! |


In [6]:
-- T4: Гейтингова алгебра на открытых подмножествах X={0..4}

newtype OpenAlg = OpenAlg { getOA :: OpenSet }
  deriving (Eq, Ord, Show)

fullSpace :: OpenSet
fullSpace = Set.fromList [0,1,2,3,4]

instance HeytingAlgebra OpenAlg where
  htop = OpenAlg fullSpace
  hbot = OpenAlg Set.empty
  hmeet (OpenAlg a) (OpenAlg b) = OpenAlg (Set.intersection a b)
  hjoin (OpenAlg a) (OpenAlg b) = OpenAlg (Set.union a b)
  himplies (OpenAlg a) (OpenAlg b) =
    OpenAlg (Set.union (Set.difference fullSpace a) b)

checkHeytingAxioms :: OpenAlg -> OpenAlg -> OpenAlg -> [(String, Bool)]
checkHeytingAxioms a b c =
  [ ("a => a = top",        himplies a a == htop)
  , ("top => a = a",        himplies htop a == a)
  , ("a /\\ (a => b) <= b",  hmeet a (himplies a b) == hmeet a b)
  , ("adjunction (=> right adjoint to /\\)",
      himplies a (himplies b c) == himplies (hmeet a b) c)
  , ("bot => a = top",      himplies hbot a == htop)
  , ("a <= neg neg a",      hmeet a (hneg (hneg a)) == a)
  , ("excl. middle (may fail)", hjoin a (hneg a) == htop)
  ]

let a = OpenAlg (Set.fromList [0,1])
let b = OpenAlg (Set.fromList [1,2])
let c = OpenAlg (Set.fromList [2,3])
putStrLn "\x2705 T4: Гейтингова алгебра на Open(X):"
mapM_ (\(s,v) -> putStrLn $ "  " ++ s ++ ": " ++ show v) (checkHeytingAxioms a b c)
let negNegA = hneg (hneg a)
putStrLn $ "a = " ++ show a ++ ", neg(neg(a)) = " ++ show negNegA
putStrLn $ "neg neg a = a: " ++ show (negNegA == a) ++ " <- в Sh(X) не всегда!"

✅ T4: Гейтингова алгебра на Open(X):

  a => a = top: True
  top => a = a: True
  a /\ (a => b) <= b: True
  adjunction (=> right adjoint to /\): True
  bot => a = top: True
  a <= neg neg a: True
  excl. middle (may fail): True

a = OpenAlg {getOA = fromList [0,1]}, neg(neg(a)) = OpenAlg {getOA = fromList [0,1]}

neg neg a = a: True <- в Sh(X) не всегда!

In [7]:
-- T4b: связки как классифицирующие стрелки (Set-топос, Omega = Bool)
-- Каждая связка = характеристическая стрелка своего определяющего подобъекта.

type Om = Bool
omElems :: [Om]
omElems = [False, True]

trueArr :: () -> Om
trueArr () = True
falseArr :: () -> Om
falseArr () = False

-- chi_S : A -> Omega ; chi_S a = (a принадлежит S), подобъект S задан списком элементов
chiOf :: Eq a => [a] -> a -> Om
chiOf s a = a `elem` s

prod2 :: [(Om, Om)]
prod2 = [(x,y) | x <- omElems, y <- omElems]

-- AND = chi <true,true> : 1 |-> Omega x Omega
andArr :: (Om, Om) -> Om
andArr = chiOf [(True, True)]

-- внутренний порядок (<=) = эквалайзер (AND, pi1) = {(x,y) | x/\y = x}
leqSub :: [(Om, Om)]
leqSub = [p | p <- prod2, andArr p == fst p]

-- IMP = chi (<=)
impArr :: (Om, Om) -> Om
impArr = chiOf leqSub

-- OR = chi образа объединения двух прямых истинности = {(x,y) | x=T или y=T}
orSub :: [(Om, Om)]
orSub = [p | p <- prod2, fst p || snd p]
orArr :: (Om, Om) -> Om
orArr = chiOf orSub

-- NOT = chi false : 1 |-> Omega ; равносильно imp . <id, false>
notArr :: Om -> Om
notArr = chiOf [False]
notViaImp :: Om -> Om
notViaImp x = impArr (x, falseArr ())

leqB :: Om -> Om -> Bool
leqB p q = (p && q) == p

pad :: Om -> String
pad b = let s = show b in s ++ replicate (5 - length s) ' '

demoArrowLogic :: IO ()
demoArrowLogic = do
  putStrLn "x     y     | and   or    imp   | classic &&/||/->"
  mapM_ (\(x,y) -> putStrLn $
            pad x ++ " " ++ pad y ++ " | "
            ++ pad (andArr (x,y)) ++ " " ++ pad (orArr (x,y)) ++ " " ++ pad (impArr (x,y))
            ++ " | " ++ show (x && y) ++ " " ++ show (x || y) ++ " " ++ show (not x || y)) prod2
  putStrLn $ "neg via chi(false): " ++ show (map notArr omElems)
  putStrLn $ "neg via (=> false): " ++ show (map notViaImp omElems)
  putStrLn $ "two negations agree: " ++ show (map notArr omElems == map notViaImp omElems)
  let adj a = and [ leqB x (impArr (a,y)) == leqB (andArr (x,a)) y
                  | x <- omElems, y <- omElems ]
  putStrLn $ "\x2705 Heyting adjunction (-/\\a -| a=>-) holds for all a: " ++ show (all adj omElems)

demoArrowLogic

Line 34: Use uncurry
Found:
fst p || snd p
Why not:
uncurry (||) p

x     y     | and   or    imp   | classic &&/||/->
False False | False False True  | False False True
False True  | False True  True  | False True True
True  False | False True  False | False True False
True  True  | True  True  True  | True True True
neg via chi(false): [True,False]
neg via (=> false): [True,False]
two negations agree: True
✅ Heyting adjunction (-/\a -| a=>-) holds for all a: True

---

## 5️⃣ T5 — Пучки и Битопосы

![Двойной классификатор](../diagrams/topos/bitopos_classifier.svg)

### Пучки на битопологическом пространстве

Пусть $(X, \tau_1, \tau_2)$ — битопологическое пространство. Тогда:

- $\mathbf{Sh}(X, \tau_1)$ — топос с классификатором $\Omega_1 = \mathcal{O}(\tau_1)$
- $\mathbf{Sh}(X, \tau_2)$ — топос с классификатором $\Omega_2 = \mathcal{O}(\tau_2)$
- Пересечение $\mathbf{BiSh}(X, \tau_1, \tau_2)$ — объекты, являющиеся пучками обоих — **не всегда топос**

### Два классификатора: битопос

**Определение.** **Битопос** $(\mathcal{E}, \Omega_+, \Omega_-)$ — категория с конечными пределами и двумя классификаторами: $\Omega_+$ (свидетельства истинности) и $\Omega_-$ (свидетельства ложности).

| | Один классификатор | Два классификатора |
|-|------|------|
| $\neg(A \wedge \neg A)$ | Аксиома | Может нарушаться |
| $A \vee \neg A$ | Необязательно | Может выполняться |
| Миров истинности | Один | Два: $\Omega_+$ и $\Omega_-$ |

### Связь с субъективным моделированием

В $\mathbf{SubjectiveModeling.ipynb}$ (разделы S13–S14) функции $\mathrm{Pl}$ и $\mathrm{Bel}$ — это обобщённые характеристические функции в двух различных топологических структурах:
$\mathrm{Pl}(A) = \mathrm{Lan}_J(\tau)(A) \leftrightarrow \Omega_+, \quad \mathrm{Bel}(A) = \mathrm{Ran}_{J^{\complement}}(\bar\tau)(A) \leftrightarrow \Omega_-$

(Правое расширение — вдоль профунктора **дополнения** $J^{\complement}$ и плотности доверия $\bar\tau$; доказательство и машинная проверка — [PytevIso.ipynb](PytevIso.ipynb).)

In [8]:
-- T5: Битопос — два классификатора с DataKinds
data Sign = Plus | Minus

-- Типизированный классификатор с полярностью (GADT + DataKinds)
data Omega (s :: Sign) h where
  OmPlus  :: HeytingAlgebra h => h -> Omega Plus h
  OmMinus :: HeytingAlgebra h => h -> Omega Minus h

getOmVal :: Omega s h -> h
getOmVal (OmPlus  h) = h
getOmVal (OmMinus h) = h

-- Битопос над [0,1] (модель через Double)
newtype Interval = UI { unUI :: Double }
  deriving (Eq, Ord)

instance Show Interval where
  show (UI x) = show (fromIntegral (round (x*100)) / 100.0 :: Double)

instance HeytingAlgebra Interval where
  htop = UI 1.0
  hbot = UI 0.0
  hmeet (UI a) (UI b) = UI (min a b)
  hjoin (UI a) (UI b) = UI (max a b)
  himplies (UI a) (UI b)
    | a <= b    = UI 1.0
    | otherwise = UI b

-- Субъективная модель: Pl = Omega+, Bel = Omega-
data PytevBT a = PytevBT
  { pytevPl  :: a -> Omega Plus  Interval
  , pytevBel :: a -> Omega Minus Interval
  }

-- Пример: tau(x) = 1 - |x-2|/3, J(x) = x < 5
exampleBT :: PytevBT Bool
exampleBT = PytevBT
  { pytevPl  = \a -> OmPlus  (UI (if a
      then maximum (map tau (filter jmap dom) ++ [0.0])
      else maximum (map tau (filter (not . jmap) dom) ++ [0.0])))
  , pytevBel = \a -> OmMinus (UI (if a
      then minimum (map tau (filter jmap dom) ++ [1.0])
      else minimum (map tau (filter (not . jmap) dom) ++ [1.0])))
  }
  where
    dom  = [0..9] :: [Int]
    tau x = max 0.0 (1.0 - abs (fromIntegral x - 4.0) / 5.0)
    jmap x = x < 5

let plT  = unUI (getOmVal (pytevPl  exampleBT True))
let belT = unUI (getOmVal (pytevBel exampleBT True))
putStrLn $ "\x2705 T5: BiTopos (Субъективная модель Пытьева):"
putStrLn $ "  Pl(True) = Omega+ = " ++ show plT
putStrLn $ "  Bel(True) = Omega- = " ++ show belT
putStrLn $ "  Bel <= Pl (закон BiTopos): " ++ show (belT <= plT)

✅ T5: BiTopos (Субъективная модель Пытьева):

  Pl(True) = Omega+ = 1.0

  Bel(True) = Omega- = 0.19999999999999996

  Bel <= Pl (закон BiTopos): True

---

## 6️⃣ T6 — Геометрические Морфизмы и Расширения Кана

![Геометрический морфизм](../diagrams/topos/geom_morph.svg)

### Определение

**Геометрический морфизм** $f : \mathcal{F} \to \mathcal{E}$ — пара функторов:
$$f^* : \mathcal{E} \to \mathcal{F} \quad \text{(обратный образ, левый сопряжённый)}$$
$$f_* : \mathcal{F} \to \mathcal{E} \quad \text{(прямой образ, правый сопряжённый)}$$
с сопряжением $f^* \dashv f_*$ и условием, что $f^*$ **сохраняет конечные пределы**.

### Связь с расширениями Кана

Для непрерывного $f : X \to Y$:

- **Прямой образ** $f_* \mathcal{F}(V) = \mathcal{F}(f^{-1}(V))$ — это **правое расширение Кана** $\mathrm{Ran}_f$
- **Обратный образ** $f^* \mathcal{G}(U) = \varinjlim_{V \supseteq f(U)} \mathcal{G}(V)$ — это **левое расширение Кана** $\mathrm{Lan}_f$

Таким образом, геометрический морфизм $f^* \dashv f_*$ — это в точности пара $(\mathrm{Lan}_f, \mathrm{Ran}_f)$ на уровне пучков. Прямая связь с $\mathbf{KanExtensions.ipynb}$.

### 2-Категория топосов

Топосы и геометрические морфизмы образуют **2-категорию** $\mathbf{Topoi}$. Терминальный объект — $\mathbf{Set}$. Морфизм глобальных сечений $\Gamma : \mathcal{E} \to \mathbf{Set}$: $\Gamma(\mathcal{F}) = \mathcal{E}(1, \mathcal{F})$.

| Расширения Кана | Геом. морфизмы |
|---------------|----------|
| $\mathrm{Lan}_f$ | $f^* : \mathbf{Sh}(Y) \to \mathbf{Sh}(X)$ |
| $\mathrm{Ran}_f$ | $f_* : \mathbf{Sh}(X) \to \mathbf{Sh}(Y)$ |
| $f^* \dashv f_*$ | Сопряжение в топосе |
| $\mathrm{Lan}_\pi$ | $\exists$-квантор |
| $\mathrm{Ran}_\pi$ | $\forall$-квантор |


In [9]:
-- T6: Геометрические морфизмы = (Lan, Ran) на пучках
-- Пучок = Map OpenSet v

type SheafMap v = Map.Map OpenSet v

-- Прямой образ: f_*(F)(V) = F(f^{-1}(V)) = Ran_f
directImage :: (OpenSet -> OpenSet) -> SheafMap v -> SheafMap v
directImage preimage sheaf =
  Map.fromList [(v, maybe err id (Map.lookup (preimage v) sheaf))
               | v <- Map.keys sheaf]
  where err = error "no section at preimage"

-- Обратный образ: f^*(G)(U) = colim G(V) для V => f(U) = Lan_f
-- (colimit в дискретном случае = значение в наименьшем V, содержащем f(U))
inverseImage :: (OpenSet -> OpenSet) -> SheafMap v -> SheafMap v
inverseImage fMap sheaf =
  Map.fromList [(u, maybe err id (Map.lookup (fMap u) sheaf))
               | u <- Map.keys sheaf]
  where err = error "no section at fMap u"

-- Пример: X = {0,1,2}, f(0)=f(1)=0, f(2)=1
let u0 = Set.fromList [0], u1 = Set.fromList [1], u01 = Set.fromList [0,1], u2 = Set.fromList [2]
let fSheaf = Map.fromList [(u0, 10::Int), (u1, 20), (u2, 30), (u01, 15)] :: SheafMap Int
let fPre v
      | v == Set.fromList [0]   = Set.fromList [0,1]
      | v == Set.fromList [1]   = Set.fromList [2]
      | v == Set.fromList [0,1] = Set.fromList [0,1,2]
      | otherwise               = Set.empty
let direct = directImage fPre fSheaf
putStrLn $ "\x2705 T6: f_*(F)({0}) = F(f^{-1}({0})) = F({0,1}) = " ++ show (Map.lookup u0 direct)
putStrLn $ "       f_*(F)({1}) = F(f^{-1}({1})) = F({2}) = "   ++ show (Map.lookup u1 direct)
putStrLn "   f_* = Ran_f, f^* = Lan_f: геометрический морфизм = расширения Кана распадаются."

---

## 7️⃣ T7 — Внутренний Язык Mitchell-Benabou

![Треугольник топос-тип-логика](../diagrams/topos/mitchell_benabou.svg)

Каждый топос $\mathcal{E}$ порождает **внутренний язык** (Mitchell 1972, Benabou 1975) — локальную интуиционистскую теорию типов с зависимыми типами.

### Словарь: топос ↔ теория типов

| Теория типов | Топос $\mathcal{E}$ |
|------------|---------------------|
| Тип $A$ | Объект $A \in \mathrm{ob}\,\mathcal{E}$ |
| Контекст $\Gamma = (x_1:A_1, \ldots, x_n:A_n)$ | $\Gamma = A_1 \times \cdots \times A_n$ |
| Терм $\Gamma \vdash t : A$ | Морфизм $t : \Gamma \to A$ |
| Предикат $\phi : \Gamma \vdash \mathrm{Prop}$ | Морфизм $\phi : \Gamma \to \Omega$ |
| Равенство $s = t$ | $\langle s,t\rangle : \Gamma \to A \times A$ через диагональ |
| Подтип $\{x:A \mid \phi(x)\}$ | Pullback $\phi^*(\mathrm{true}) \hookrightarrow A$ |

### Квантификаторы как расширения Кана

Квантификаторы определяются **вдоль проекции** $\pi : \Gamma \times A \to \Gamma$.
Для предиката $\phi : \Gamma \times A \to \Omega$:
$$\forall x:A.\, \phi(x) = \mathrm{Ran}_\pi(\phi) : \Gamma \to \Omega$$
$$\exists x:A.\, \phi(x) = \mathrm{Lan}_\pi(\phi) : \Gamma \to \Omega$$

В $\mathbf{Set}$ это даёт обычные $\forall$ и $\exists$. В $\mathbf{Sh}(X)$ — общее понятие: $\exists$ через пучкование colimit, $\forall$ — через limit.

### Связь с Haskell

| Haskell | Mitchell-Benabou | Категорно |
|---------|-----------------|----------|
| `forall a. F a` | $\forall x:A.\, F(x)$ | $\mathrm{Ran}_\pi(F)$ |
| `exists a. F a` | $\exists x:A.\, F(x)$ | $\mathrm{Lan}_\pi(F)$ |
| `a -> b` | $A \Rightarrow B$ | $B^A$ (степенной) |
| `a -> Bool` | $\phi : A \to \Omega$ | Морфизм в $\Omega$ |
| `[x | p x]` | $\{x:A \mid \phi(x)\}$ | Pullback |

### Теорема корректности (Seely, 1984)

Внутренний язык $\mathcal{L}(\mathcal{E})$ топоса $\mathcal{E}$ — это **интуиционистская локальная теория типов высшего порядка**. Существует биекция (до изоморфизма) между элементарными топосами и интуиционистскими локальными теориями типов высшего порядка.


In [10]:
-- T7: Внутренний язык Mitchell-Benabou в Haskell
-- Терм в контексте ctx о типе a = морфизм ctx -> a в топосе
-- (Term ctx a = Term ctx a уже введён в T0)

-- Логические связки как морфизмы в Omega
andPred :: Predicate ctx -> Predicate ctx -> Predicate ctx
andPred (Term f) (Term g) = Term (\c -> f c && g c)

orPred :: Predicate ctx -> Predicate ctx -> Predicate ctx
orPred (Term f) (Term g) = Term (\c -> f c || g c)

impliesPred :: Predicate ctx -> Predicate ctx -> Predicate ctx
impliesPred (Term f) (Term g) = Term (\c -> not (f c) || g c)

negPred :: Predicate ctx -> Predicate ctx
negPred p = impliesPred p (Term (\_ -> False))

-- Подтип путём pullback phi^(true)
comprehension :: Predicate a -> [a] -> [a]
comprehension (Term phi) = filter phi

-- KVANTIFIKATORY KAK RASSHIRENIYA KANA
-- forall = Ran_pi: (Gamma x A -> Omega) -> (Gamma -> Omega)
forallKan :: [a] -> (ctx -> a -> Bool) -> Predicate ctx
forallKan dom phi = Term (\g -> all (phi g) dom)

-- exists = Lan_pi: (Gamma x A -> Omega) -> (Gamma -> Omega)
existsKan :: [a] -> (ctx -> a -> Bool) -> Predicate ctx
existsKan dom phi = Term (\g -> any (phi g) dom)

-- Пример: Gamma = Int (переменная n), A = Int, phi(n,m) = m < n
let dom0 = [0..10] :: [Int]
let phiLt n m = m < n
let forallLt = forallKan dom0 phiLt
let existsLt = existsKan dom0 phiLt

putStrLn "\x2705 T7: Квантификаторы = расширения Кана (n: Gamma=Int, m: A=Int):"
putStrLn $ "  forall m in [0..10]. m < 3 = Ran_pi(phi)(3) = " ++ show (runTerm forallLt 3)
putStrLn $ "  exists m in [0..10]. m < 3 = Lan_pi(phi)(3) = " ++ show (runTerm existsLt 3)
putStrLn $ "  exists m in [0..10]. m < 0 = " ++ show (runTerm existsLt 0)
let subtype = comprehension existsLt [0..10]
putStrLn $ "  {n in [0..10] | exists m. m<n} = " ++ show subtype ++ " <- pullback phi^(true)"
-- forall = Ran (единственный способ продолжить стрелку ctx->Omega через проекцию)
putStrLn "  forall = Ran_pi (right Kan), exists = Lan_pi (left Kan) -- Mitchell-Benabou theorem"

✅ T7: Квантификаторы = расширения Кана (n: Gamma=Int, m: A=Int):

  forall m in [0..10]. m < 3 = Ran_pi(phi)(3) = False

  exists m in [0..10]. m < 3 = Lan_pi(phi)(3) = True

  exists m in [0..10]. m < 0 = False

  {n in [0..10] | exists m. m<n} = [1,2,3,4,5,6,7,8,9,10] <- pullback phi^(true)

  forall = Ran_pi (right Kan), exists = Lan_pi (left Kan) -- Mitchell-Benabou theorem

---

## 8️⃣ T8 — Итоги: одна конструкция в трёх топосах

Классификатор подобъектов $\Omega$ и его внутренняя логика — это **один** аппарат,
по-разному реализованный в разных топосах. Ниже — что становится истиной, отрицанием,
кванторами в каждом из них.

| Конструкция | $\mathbf{Set}$ | $\mathbf{Sh}(X)$ | $\mathbf{Sub.Mod.}$ |
|-------------|---------|-----------|----------------------|
| $\Omega$ | $\{\bot,\top\}$ | $\mathcal{O}(-)$ | $[0,1]$ |
| $\mathrm{true}$ | $\top$ | $U \mapsto U$ | $1.0$ |
| Характ. функция | $A \to \{0,1\}$ | $\mathcal{F} \to \Omega_X$ | $\tau : X \to [0,1]$ |
| $\neg$ | Дополнение | $\mathrm{Int}(X \setminus -)$ | $1-(-)$ |
| $\neg\neg$ | $\mathrm{id}$ | $\mathrm{Int}(\mathrm{Cl}(-))$ | Не обязат. |
| $\forall$ | $\bigcap$ | $\mathrm{Ran}_\pi$ | $\mathrm{Ran}_{J^{\complement}}(\bar\tau) = \mathrm{Bel}$ |
| $\exists$ | $\bigcup$ | $\mathrm{Lan}_\pi$ | $\mathrm{Lan}_J(\tau) = \mathrm{Pl}$ |
| Геом. морф. | Функция | Непрерывное | Расширение Кана |
| Логика | Классич. | Интуиц. | Субъект. |

Один и тот же $\Omega$, одни и те же связки-стрелки (раздел T4) — меняется лишь топос, в
котором они интерпретируются: от классической логики ($\mathbf{Set}$) к интуиционистской
($\mathbf{Sh}(X)$) и к субъектной ($[0,1]$-обогащение).

> Карта всего курса, граф зависимостей между ноутбуками и статусы разработки — в
> [README.ipynb](../README.ipynb) и `ROADMAP.md` (здесь, в учебном разделе, им не место).

Код ниже верифицирует ключевые факты по всем трём топосам.

In [11]:
-- T8: Сводная верификация
-- 1. Omega в Set = Bool
let boolOK = and
      [ himplies True  True  == True
      , himplies False True  == True
      , himplies True  False == False
      , hmeet True (hneg True) == False
      , hjoin True (hneg True) == True
      , all (\b -> hneg (hneg b) == b) [True, False]
      ]
putStrLn $ "\x2705 T8: Bool (Set topos): " ++ show boolOK

-- 2. Omega в Sh(X) = OpenAlg
let v = OpenAlg (Set.fromList [0,1])
let emOK = hjoin v (hneg v) == htop
let nnOK = hneg (hneg v) == v
putStrLn $ "  OpenAlg excluded middle: " ++ show emOK ++ " (в Sh(X) не всегда)"
putStrLn $ "  OpenAlg neg neg = id: "    ++ show nnOK  ++ " (в Sh(X) не всегда)"

-- 3. forall = Ran, exists = Lan (верификация адъюнкции)
let dom1 = [0..5] :: [Int]
let phi1 n m = m < n
let fa = runTerm (forallKan dom1 phi1) 3
let ex = runTerm (existsKan dom1 phi1) 3
putStrLn $ "  forall m in [0..5]. m<3: " ++ show fa ++ " (Ran_pi)"
putStrLn $ "  exists m in [0..5]. m<3: " ++ show ex ++ " (Lan_pi)"

-- 4. BiTopos: Bel <= Pl
let pts = [0.0,0.5,1.0,1.5,2.0,2.5,3.0,4.0] :: [Double]
let plF x = max 0.0 (1.0 - abs (x-2.0)/3.0)
let belF x = max 0.0 (1.0 - abs (x-2.0)/1.0)
let biOK = all (\x -> belF x <= plF x) pts
putStrLn $ "  Bel <= Pl (BiTopos law): " ++ show biOK
putStrLn ""
putStrLn "\x2705 T8: Все основные теоремы верифицированы."

✅ T8: Bool (Set topos): True

  OpenAlg excluded middle: True (в Sh(X) не всегда)

  OpenAlg neg neg = id: True (в Sh(X) не всегда)

  forall m in [0..5]. m<3: False (Ran_pi)

  exists m in [0..5]. m<3: True (Lan_pi)

  Bel <= Pl (BiTopos law): True

✅ T8: Все основные теоремы верифицированы.

---

## 9️⃣ T9 — Вишенка: Категория $\mathbf{BiTopos}$

![Зоопарк топосов](../diagrams/topos/bitopos_zoo.svg)

### Определение

**Определение (BiTopos).** **Битопос** — это тройка $(\mathcal{E}, \Omega_+, \Omega_-)$, где:

1. $\mathcal{E}$ — категория с конечными пределами
2. $\Omega_+ \in \mathcal{E}$ с $\mathrm{true}_+ : 1 \to \Omega_+$ — **положительный классификатор** (свидетельства истинности)
3. $\Omega_- \in \mathcal{E}$ с $\mathrm{true}_- : 1 \to \Omega_-$ — **отрицательный классификатор** (свидетельства ложности)

**Морфизм** $f : (\mathcal{F}, \Omega_+^\mathcal{F}, \Omega_-^\mathcal{F}) \to (\mathcal{E}, \Omega_+^\mathcal{E}, \Omega_-^\mathcal{E})$ — геометрический морфизм $(f^*, f_*)$ с морфизмами $f^*(\Omega_\pm^\mathcal{E}) \to \Omega_\pm^\mathcal{F}$, совместимыми с $\mathrm{true}_\pm$.

### Почему два классификатора = две логики

$\Omega_+$ порождает логику **свидетельств истинности**: $\neg_+ \phi = \phi \Rightarrow_{\Omega_+} \bot_+$.
$\Omega_-$ порождает логику **свидетельств ложности**: $\neg_- \phi = \phi \Rightarrow_{\Omega_-} \bot_-$.

**Паранепротиворечивость:** $\neg_+(\phi \wedge_+ \neg_- \phi)$ **не является аксиомой** — объект может иметь свидетельства и истины, и лжи одновременно.
**Параполнота:** $\phi \vee_+ \neg_+ \phi$ **не является аксиомой** — может не быть ни свидетельств, ни против (неполнота).

### Зоопарк топосов

| Топос | $\Omega_+$ | $\Omega_-$ | Логика |
|------|------------|------------|--------|
| $\mathbf{Set}$ | $\{\bot,\top\}$ | — | Классическая |
| $\mathbf{Sh}(X)$ | $\mathcal{O}(\tau)$ | — | Интуиц. |
| $\mathbf{Set}^2$ (битопос) | $\{\bot,\top\}$ | $\{\bot,\top\}$ | 4-значная де Моргана |
| $\mathbf{BiSh}(X,\tau_1,\tau_2)$ | $\mathcal{O}(\tau_1)$ | $\mathcal{O}(\tau_2)$ | Параинтуиц. |
| $\mathbf{BiTopos}([0,1])$ | $\mathrm{Pl} = \mathrm{Lan}_J(\tau)$ | $\mathrm{Bel} = \mathrm{Ran}_{J^{\complement}}(\bar\tau)$ | Субъективная |

### Почему это работает

В $\mathbf{SubjectiveModeling.ipynb}$ функции $\mathrm{Pl}$ и $\mathrm{Bel}$ — это обобщённые характеристические функции в quantale-обогащённом BiTopos. $\mathrm{Bel} \leq \mathrm{Pl}$ — не автоматизм порядка расширений, а следствие **дуальной согласованности и нормировки**: $\mathrm{Bel} = \theta \circ \mathrm{Pl} \circ \complement$, а на tot-парах d-меры согласованность выводится чисто порядково (теоремы и машинная проверка — [PytevIso.ipynb](PytevIso.ipynb)). Это **фундаментальное объяснение** дуальности через BiTopos.

### Перспективы

- **$\omega$-BitTopos**: битопосы со счётными операциями — категорное основание теории меры
- **$(n,1)$-BitTopos**: $n$-категорные аналоги (теория $\infty$-пучков с двойной логикой)
- **Обогащённые BitTopoi**: над quantaloid — обобщение Lawvere до параинтуиционизма

In [12]:
-- T9: Категория BiTopos — строгая реализация с GADTs + DataKinds

-- Расширяем Sign и Omega из T5
-- Omega Plus/Minus уже есть; строим BiTopos-структуру

-- BiTopos: объект A с двумя характеристическими стрелками
data BiSub a h = BiSub
  { posChar :: a -> Omega Plus  h   -- A -> Omega+
  , negChar :: a -> Omega Minus h   -- A -> Omega-
  }

-- Проверка: закон BiTopos: Omega-(a) <= Omega+(a)
biToposLaw :: Ord h => BiSub a h -> [a] -> Bool
biToposLaw sub xs = all (\a -> getOmVal (negChar sub a) <= getOmVal (posChar sub a)) xs

-- Полная модель субъективности Пытьева как BiTopos над [0,1]
pytevFullModel :: (Int -> Double) -> (Int -> Bool) -> [Int] -> BiSub Bool Interval
pytevFullModel tau jmap dom = BiSub
  { posChar = \a -> OmPlus (UI (calcPl a))
  , negChar = \a -> OmMinus (UI (calcBel a))
  }
  where
    matching a = if a then filter jmap dom else filter (not . jmap) dom
    calcPl  a = maximum (map tau (matching a) ++ [0.0])
    tauBar x  = 1.0 - tau x
    calcBel a = minimum (map tauBar (matching (not a)) ++ [1.0])

-- Пример: X = {0..9}, tau(x) = 1 - |x-4|/5, J(x) = x < 5
let dom9 = [0..9] :: [Int]
let tau9 x = max 0.0 (1.0 - abs (fromIntegral x - 4.0) / 5.0)
let j9    = (< 5) :: Int -> Bool
let pytev = pytevFullModel tau9 j9 dom9

let plT  = unUI (getOmVal (posChar pytev True))
let belT = unUI (getOmVal (negChar pytev True))
let plF  = unUI (getOmVal (posChar pytev False))
let belF = unUI (getOmVal (negChar pytev False))
putStrLn "\x2705 T9: BiTopos Пытьева (полная модель):"
putStrLn $ "  Pl(True)  = Omega+ = " ++ show plT
putStrLn $ "  Bel(True) = Omega- = " ++ show belT
putStrLn $ "  Pl(False) = Omega+ = " ++ show plF
putStrLn $ "  Bel(False)= Omega- = " ++ show belF
putStrLn $ "  Bel <= Pl (True):  " ++ show (belT <= plT)
putStrLn $ "  Bel <= Pl (False): " ++ show (belF <= plF)
putStrLn $ "  BiTopos law holds: " ++ show (biToposLaw pytev [True, False])
putStrLn ""
putStrLn "Зоопарк топосов:"
putStrLn "  Set         -> Omega={F,T}       -> classical logic"
putStrLn "  Sh(X)       -> Omega=Open(X)     -> intuitionistic logic"
putStrLn "  BiTopos(Set)-> Omega+={F,T} Omega-={F,T} -> de Morgan 4-valued"
putStrLn "  BiTopos([0,1])-> Omega+=Pl Omega-=Bel    -> Pytev subjective logic"

✅ T9: BiTopos Пытьева (полная модель):

  Pl(True)  = Omega+ = 1.0

  Bel(True) = Omega- = 0.19999999999999996

  Pl(False) = Omega+ = 0.8

  Bel(False)= Omega- = 0.0

...

Зоопарк топосов:

  Set         -> Omega={F,T}       -> classical logic

  Sh(X)       -> Omega=Open(X)     -> intuitionistic logic

  BiTopos(Set)-> Omega+={F,T} Omega-={F,T} -> de Morgan 4-valued

  BiTopos([0,1])-> Omega+=Pl Omega-=Bel    -> Pytev subjective logic

---

## T9 — Haskell и топосная структура: что есть в коде

Топосная теория проявляется в Haskell на нескольких уровнях — от алгебры значений истинности до структуры оптических библиотек.

### Алгебры Хейтинга в Haskell

`Bool` — булева алгебра (классический топос $\mathbf{Set}$): закон исключённого третьего $a \vee \neg a = \top$ всегда выполняется. Для интуиционистской логики нужна **неклассическая алгебра Хейтинга** — решётка с импликацией $a \Rightarrow b$, где $\neg\neg a \neq a$ в общем случае. Алгебра Лукасевича над $[0,1]$ — пример такой неклассической структуры.

### Предпучки как функторы

`newtype Presheaf c a = Presheaf { sections :: c -> [a] }` — любой функтор $c \to \mathbf{Set}$ является предпучком. Контравариантность (функтор $\mathcal{C}^{\mathrm{op}} \to \mathbf{Set}$) моделируется через противоположную категорию или обёртку `Op`.

### Классификатор $\Omega$

В $\mathbf{Set}$ это `Bool`. В категории предпучков $[\mathcal{C}, \mathbf{Set}]$ классификатор — это `Sub` (множество подобъектов): `Hom(A, Omega) ~= Sub(A)`. Характеристическая функция `chi :: (a -> Bool) -> a -> Omega` — прямое воплощение этого изоморфизма.

### Traversable как топосная структура

`traverse` — это морфизм пучков: он сохраняет структуру контейнера, меняя значения с аппликативным эффектом. Закон согласованности `traverse (Const . f) = Const . foldMap f` — это условие, что секции над разными открытыми множествами склеиваются корректно.

### Lens как внутренний язык

`Lens s a = forall f. Functor f => (a -> f a) -> s -> f s` — это форма зависимого типа в стиле Mitchell–Benabou: `Lens` параметризован функтором `f` так же, как внутренний язык топоса параметризован объектами $\Omega$. `view` соответствует извлечению секции, `over` — морфизму пучков.

In [13]:
-- Алгебра Хейтинга: обобщение булевой алгебры
-- В топосе каждый объект Omega несёт структуру алгебры Хейтинга.
-- Используем отдельный класс (не HeytingAlgebra из T0) во избежание конфликта.
class (Ord a) => HAlg a where
  ha_top  :: a              -- истина (1)
  ha_bot  :: a              -- ложь (0)
  ha_meet :: a -> a -> a    -- конъюнкция (meet, ∧)
  ha_join :: a -> a -> a    -- дизъюнкция (join, ∨)
  ha_imp  :: a -> a -> a    -- импликация (→)
  ha_neg  :: a -> a
  ha_neg x = ha_imp x ha_bot

-- Bool — булева алгебра (классический топос Set)
instance HAlg Bool where
  ha_top  = True
  ha_bot  = False
  ha_meet = (&&)
  ha_join = (||)
  ha_imp a b = not a || b

-- [0,1] как алгебра Лукасевича (интуиционистская)
newtype UnitH = UnitH Double deriving (Eq, Ord)
instance Show UnitH where show (UnitH x) = show (fromIntegral (round (x*100)) / 100.0 :: Double)

instance HAlg UnitH where
  ha_top = UnitH 1.0
  ha_bot = UnitH 0.0
  ha_meet (UnitH a) (UnitH b) = UnitH (min a b)
  ha_join (UnitH a) (UnitH b) = UnitH (max a b)
  ha_imp  (UnitH a) (UnitH b) = UnitH (min 1.0 (1.0 - a + b))  -- Lukasiewicz

-- Предпучок как (контравариантный) функтор c -> [a]
newtype Presheaf c a = Presheaf { sections2 :: c -> [a] }

-- Классификатор подобъектов Omega в Hask ~ Bool
type OmegaSet = Bool
chiOmega :: (a -> Bool) -> a -> OmegaSet
chiOmega = id

-- Демонстрация: закон исключённого третьего
let emBool  x = ha_join x (ha_neg x) == ha_top
let emUnit  x = ha_join x (ha_neg x) == ha_top

putStrLn "\x2705 T9 (new): Алгебра Хейтинга"
putStrLn $ "  Bool: excluded middle True  = " ++ show (emBool True)
putStrLn $ "  Bool: excluded middle False = " ++ show (emBool False)
putStrLn $ "  Bool: neg(neg(False)) == False: " ++ show (ha_neg (ha_neg False) == False)

let u03 = UnitH 0.3
let u07 = UnitH 0.7
putStrLn $ "  Lukasiewicz [0,1]: neg(neg(0.3)) = " ++ show (ha_neg (ha_neg u03)) ++ " (should be 0.3)"
putStrLn $ "  Lukasiewicz [0,1]: 0.3 => 0.7    = " ++ show (ha_imp u03 u07)
putStrLn $ "  Lukasiewicz [0,1]: 0.7 => 0.3    = " ++ show (ha_imp u07 u03)
-- In Lukasiewicz: neg(neg(x)) = x (it IS a De Morgan algebra, so double negation holds there)
-- But excluded middle fails:
let em03 = ha_join u03 (ha_neg u03)
putStrLn $ "  Lukasiewicz: 0.3 \\/  neg(0.3) = " ++ show em03 ++ " /= 1.0 <- not excluded middle"
-- Presheaf example: sections over a discrete base category {0,1,2}
let psh = Presheaf (\n -> replicate n '*') :: Presheaf Int Char
putStrLn $ "  Presheaf(2) = " ++ show (sections2 psh 2)
putStrLn $ "  Presheaf(0) = " ++ show (sections2 psh 0)
putStrLn $ "  chiOmega even 4 = " ++ show (chiOmega even (4::Int)) ++ ", chiOmega even 3 = " ++ show (chiOmega even (3::Int))

✅ T9 (new): Алгебра Хейтинга

  Bool: excluded middle True  = True

  Bool: excluded middle False = True

  Bool: neg(neg(False)) == False: True

  Lukasiewicz [0,1]: neg(neg(0.3)) = 0.3 (should be 0.3)

...

  Lukasiewicz [0,1]: 0.7 => 0.3    = 0.6

  Lukasiewicz: 0.3 \/  neg(0.3) = 0.7 /= 1.0 <- not excluded middle

  Presheaf(2) = "**"

  Presheaf(0) = ""

  chiOmega even 4 = True, chiOmega even 3 = False

In [14]:
-- Van Laarhoven lens: форма зависимого типа Mitchell-Benabou
-- Lens s a = forall f. Functor f => (a -> f a) -> s -> f s
-- "forall f" работает благодаря RankNTypes (включён в SETUP)
-- Это "внутренний hom" в категории функторов над топосом Set

type LensT s a = forall f. Functor f => (a -> f a) -> s -> f s

-- Const функтор = "просто прочитать" (заморозить значение)
newtype ConstL r a = ConstL { getConstL :: r } deriving (Show)
instance Functor (ConstL r) where fmap _ (ConstL r) = ConstL r
-- как Applicative (нужно для traverse): накапливаем r в моноиде
instance Monoid r => Applicative (ConstL r) where
  pure _ = ConstL mempty
  ConstL a <*> ConstL b = ConstL (a <> b)

-- Identity функтор = "просто изменить"
newtype IdentityL a = IdentityL { runIdentityL :: a } deriving (Show)
instance Functor IdentityL where fmap f (IdentityL a) = IdentityL (f a)
instance Applicative IdentityL where
  pure = IdentityL
  IdentityL f <*> IdentityL a = IdentityL (f a)

-- view и over через единый тип LensT
viewL :: LensT s a -> s -> a
viewL l s = getConstL (l ConstL s)

overL :: LensT s a -> (a -> a) -> s -> s
overL l f s = runIdentityL (l (IdentityL . f) s)

setL :: LensT s a -> a -> s -> s
setL l x = overL l (const x)

-- Пример: фокус на первом элементе пары
_fstL :: LensT (a, b) a
_fstL f (a, b) = fmap (\a' -> (a', b)) (f a)

-- Пример: фокус на втором элементе
_sndL :: LensT (a, b) b
_sndL f (a, b) = fmap (\b' -> (a, b')) (f b)

-- Traversal = морфизм пучков: traverse сохраняет структуру, меняет значения
-- traverse :: (Traversable t, Applicative f) => (a -> f b) -> t a -> f (t b)
-- Закон: traverse (Identity . id) = Identity (сохранение идентичности)
-- Const-обход = foldMap (сбор значений в моноид)
checkTraverseLaws :: IO ()
checkTraverseLaws = do
  let xs = [1,2,3,4,5] :: [Int]
  -- Закон 1: traverse (Identity . id) = Identity  =>  runIdentityL (...) == xs
  let law1 = runIdentityL (traverse (IdentityL . id) xs) == xs
  -- Const-обход собирает элементы в список (моноид), затем суммируем
  let collected      = getConstL (traverse (\x -> ConstL [x]) xs) :: [Int]
      sumViaTraverse = sum collected
      sumDirect      = sum xs
  putStrLn $ "\x2705 T9 (new): Lens как внутренний язык"
  putStrLn $ "  view _fst (10, 20)    = " ++ show (viewL _fstL (10::Int, 20::Int))
  putStrLn $ "  over _fst (*2) (3,4)  = " ++ show (overL _fstL (*2) (3::Int, 4::Int))
  putStrLn $ "  set  _snd 99  (1,2)   = " ++ show (setL  _sndL 99 (1::Int, 2::Int))
  putStrLn $ "  Traversal law (traverse (Id.id) = id on lists): " ++ show law1
  putStrLn $ "  Const-traverse collects [1..5] => sum = " ++ show sumViaTraverse ++ " == " ++ show sumDirect
  putStrLn   "  Lens = forall f. Functor f => (a->fa)->s->fs"
  putStrLn   "       = внутренний hom в Cat(Functor) = Mitchell-Benabou зависимый тип"

checkTraverseLaws

Line 35: Use tuple-section
Found:
\ a' -> (a', b)
Why not:
(, b)Line 39: Use tuple-section
Found:
\ b' -> (a, b')
Why not:
(a,)Line 49: Redundant id
Found:
IdentityL . id
Why not:
IdentityLLine 54: Redundant $
Found:
putStrLn $ "\x2705 T9 (new): Lens как внутренний язык"
Why not:
putStrLn "\x2705 T9 (new): Lens как внутренний язык"

✅ T9 (new): Lens как внутренний язык
  view _fst (10, 20)    = 10
  over _fst (*2) (3,4)  = (6,4)
  set  _snd 99  (1,2)   = (1,99)
  Traversal law (traverse (Id.id) = id on lists): True
  Const-traverse collects [1..5] => sum = 15 == 15
  Lens = forall f. Functor f => (a->fa)->s->fs
       = внутренний hom в Cat(Functor) = Mitchell-Benabou зависимый тип

## Сводная таблица: топосная теория в Haskell

| Топосное понятие | Реализация в Haskell |
|-----------------|---------------------|
| Классификатор $\Omega$ | `Bool` (в $\mathbf{Set}$), `UnitH` (в $[0,1]$-топосе Лукасевича) |
| Алгебра Хейтинга | `HAlg` typeclass; инстанции `Bool`, `UnitH` |
| Предпучок $F: \mathcal{C}^{\mathrm{op}} \to \mathbf{Set}$ | `newtype Presheaf c a = Presheaf (c -> [a])` |
| Характеристическая функция $\chi$ | `chiOmega :: (a -> Bool) -> a -> OmegaSet` |
| Внутренний hom / зависимый тип | `LensT s a = forall f. Functor f => (a -> f a) -> s -> f s` |
| `traverse` = морфизм пучков | `Traversable`, законы `traverse pure = pure`, `foldMap` |
| Геометрический морфизм $f^*$ | `fmap` (обратный образ функтора) |
| Подобъект / характер. функция | `a -> Bool` (предикат = секция над $\Omega$) |

---

Исключённое третье **выполняется** для `Bool` (классический топос), но **нарушается** для `UnitH` (интуиционистская алгебра Лукасевича): $0.3 \vee \neg 0.3 = 0.3 \vee 0.7 = 0.7 \neq 1$.

`Lens` реализует **принцип Митчелла–Бенабу**: параметрический полиморфизм по функтору `f` — это форма зависимого типа, в которой тип результата зависит от выбора «мира» (объекта-функтора) во внутреннем языке топоса.

![Haskell и топосные структуры](../diagrams/topos/topos_libraries.svg)

## Источники и якоря конструкций

> **★ = наш синтез**; без звёздочки — классика. Сводная библиотека якорей нити Пытьева — в [PytevIso.ipynb](PytevIso.ipynb).

| Конструкция | Якорь |
|---|---|
| Элементарный топос, аксиомы | Mac Lane–Moerdijk, *Sheaves in Geometry and Logic*; Johnstone, *Sketches of an Elephant* |
| Классификатор подобъектов $\Omega$, внутренняя логика, алгебры Гейтинга | Goldblatt, *Topoi: The Categorial Analysis of Logic*; Mac Lane–Moerdijk; Awodey, *Category Theory* |
| Предпучки и пучки | Mac Lane–Moerdijk, гл. I–III |
| Геометрические морфизмы | Johnstone, *Elephant*; Mac Lane–Moerdijk |
| Внутренний язык Митчелла–Бенабу | Mac Lane–Moerdijk; Johnstone |
| Битопологическая природа (d-frames) | Jung–Moshier, «On the bitopological nature of Stone duality» (2006) |
| **★ $\mathbf{BiTopos}$ (два классификатора = субъективная логика Пытьева); одна конструкция в трёх топосах** | наш синтез (связь с [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) / [PytevIso.ipynb](PytevIso.ipynb)) |

---

**← Предыдущий:** [GPU / Accelerate](GPUHaskell.ipynb)  |  **Следующий →** [Неопределённость](Uncertainty.ipynb)
